# Cross-Modal Contrastive Alignment and the Modality Gap

InfoNCE taught a single dual encoder to place a query near its answer; dense retrieval read that
geometry as a maximum-inner-product lookup. A **multimodal** system trains *two* encoders — a text
tower and a chart tower — with the same symmetric contrastive (CLIP) loss, so a text query can
retrieve a chart of the same company. Liang et al. (2022) noticed that even after training the two
modalities sit in **disjoint cones** on the sphere: a measurable *modality gap* between the text and
chart centroids.

This notebook imports the tested reference module
[`cross_modal_alignment.py`](cross_modal_alignment.py) — which **owns every number** — and walks the
three movements. The thesis: **the gap is a calibration artifact, invisible to MIPS ranking.**

In [ ]:
import pathlib, sys
import numpy as np

# Import the canonical module (it owns the numbers); it puts every prerequisite dir on sys.path.
sys.path.insert(0, str(pathlib.Path.cwd()))
import cross_modal_alignment as cma

Zt, Zc, sector, comp = cma.cross_modal_corpus()
truth = np.arange(Zt.shape[0])
print("companies:", Zt.shape[0], " dim:", Zt.shape[1], " modality gap ||g|| =", round(cma.modality_gap(Zt, Zc), 4))

## Movement 1 — what the gap *is*: an orthogonal (Pythagorean) decomposition

For paired unit embeddings $z_{\text{text},i}$ and $z_{\text{chart},i}$, write the per-pair
difference $d_i = z_{\text{text},i} - z_{\text{chart},i}$ and the **gap vector**
$g = \frac1n\sum_i d_i = \overline{z_{\text{text}}} - \overline{z_{\text{chart}}}$. Splitting the
difference matrix $D = \mathbf{1}g^\top + (D - \mathbf{1}g^\top)$ into its coherent rank-1 part and
its orthogonal complement gives the exact identity

$$L_{\text{align}} = \tfrac1n\sum_i\lVert d_i\rVert^2 = \lVert g\rVert^2 + \tfrac1n\sum_i\lVert d_i-g\rVert^2 = \text{gap}^2 + \text{dispersion}.$$

$L_{\text{align}}$ **is** the imported `alignment`; the gap is the projection of the misalignment onto
the shared-offset subspace, so $\text{gap}^2 \le L_{\text{align}}$ (a projection is a contraction).

In [ ]:
dec = cma.decompose_alignment(Zt, Zc)
print("L_align       =", round(dec["L_align"], 4), " == imported alignment:", round(cma.alignment(Zt, Zc), 4))
print("gap^2         =", round(dec["gap2"], 4))
print("dispersion    =", round(dec["dispersion"], 4))
print("gap^2 + disp  =", round(dec["gap2"] + dec["dispersion"], 4), " (== L_align)")
print("<coherent, dispersion>_F =", f'{dec["ortho"]:.2e}', " (orthogonal)")

## Movement 2 — the gap is a calibration artifact, invisible to MIPS ranking

Cross-modal scores are $S_{ij} = \langle t_i, c_j\rangle$. Offset every chart embedding by the gap,
$c_j \mapsto c_j + \alpha g$; then $S'_{ij} = S_{ij} + \alpha\langle t_i, g\rangle$, and the added
term is a **per-query constant** — so the argsort over $j$, and recall@$k$, are *exactly* invariant
to the gap under maximum-inner-product search. Cosine, which **renormalizes** $c_j + \alpha g$, is
*not* gap-invariant: its recall moves and improves as the offset removes the gap.

In [ ]:
mips = cma.mips_recall_curve(Zt, Zc, truth)
cos = cma.cosine_recall_curve(Zt, Zc, truth)
md, same = cma.mips_offset_invariance_witness(Zt, Zc, alpha=1.0)
print("alpha grid    :", list(cma.CM_ALPHA_GRID))
print("MIPS recall@1 :", [round(r, 3) for r in mips], " (FLAT; max score-dev", f"{md:.1e}", "argsort identical:", str(same) + ")")
print("cosine recall :", [round(r, 3) for r in cos], " (MOVES — peaks as the gap is removed)")
g = cma.gap_vector(Zt, Zc)
print("cross-modal cosine:", round(cma.cross_modal_cosine(Zt, Zc), 4),
      "with gap ->", round(cma.cross_modal_cosine(Zt, cma.normalize(cma.offset_keys(Zc, g, 1.0))), 4), "without")

## Movement 3 — where the gap comes from: the cone effect and temperature

A vMF toy seeds two modality cones at "initialization" (separation $\beta$). "Training" is a
deterministic full-batch projected-GD descent on the **symmetric CLIP loss** (the imported
`symmetric_inbatch_loss`). We do **not** claim training preserves the gap — to convergence,
full-batch GD *closes* it at moderate temperature. The robust, seed-stable claim is the **monotone
direction**: lower temperature preserves a larger residual gap.

In [ ]:
for r in cma.residual_gap_vs_tau():
    print(f"tau={r['tau']:.2f}: gap {r['gap_before']:.3f} -> {r['gap_after']:.3f}"
          f"   alignment={r['alignment']:.3f}   uniformity={r['uniformity']:.3f}")

## Finance case study — text filings vs price charts of the same companies

A desk embeds 10-K text and price charts of the same companies with two towers. A modality gap opens
between the cones, yet the cross-modal **ranking** (which chart a text query retrieves) is invisible
to it under MIPS, while a fixed similarity **threshold** is corrupted by it — the calibration cost.

In [ ]:
_ = cma.finance_demo()

## Verification harness and viz constants

Every pedagogical claim above is an `assert` in the module; `viz_constants()` prints the numbers the
laboratory mirrors to the decimal.

In [ ]:
cma._run_all()
print("\nViz constants:")
cma.viz_constants()
print("\nAll checks passed.")